# Build `kurucz_cd23_chianti_H_He_latest.h5`

This notebook builds a TARDIS atom-data file from the current Carsus public data inputs:

- NIST atomic weights and ionization energies for H-Zn
- Kurucz CD23 GFALL levels and lines from the [Carsus Kurucz data mirror](https://github.com/tardis-sn/carsus-data-kurucz)
- CHIANTI H-He levels, lines, and electron collision strengths from the [CHIANTI database](https://www.chiantidatabase.org/) configured by `XUVTOP`
- Knox-Long recombination zeta data
- [NNDC ENSDF CSV decay-radiation data](https://github.com/tardis-sn/carsus-data-nndc)

The output uses the legacy TARDIS HDF schema (`database_version='v0.9'`) expected by TARDIS regression atom-data files. It is intended to make a valid current-data file named `kurucz_cd23_chianti_H_He_latest.h5`; it is not a byte-for-byte reproduction of older historical regression files.

## Configure source locations

Download and extract the latest CHIANTI database from the CHIANTI project, then set `XUVTOP` to the extracted database root before running the Carsus readers. The current CHIANTI database package should contain files such as `VERSION`, `masterlist/masterlist_ions.pkl`, `h/h_1/h_1.elvlc`, and `he/he_1/he_1.elvlc`.

The optional `CHIANTI_DATABASE_URL` block below is provided for reproducible local runs when you know the exact upstream archive URL. If `XUVTOP` already points to an extracted CHIANTI tree, the notebook will use that tree and skip downloading.

In [1]:
from pathlib import Path
import os
import shutil
import tarfile
import urllib.request

DATA_ROOT = Path(os.environ.get("CARSUS_ATOMDATA_SOURCE_DIR", Path.home() / "Downloads" / "carsus-atomdata-sources"))
DATA_ROOT.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = Path("kurucz_cd23_chianti_H_He_latest.h5")

# Prefer an existing XUVTOP. If it is not set, use DATA_ROOT/chianti after optional download/extraction.
CHIANTI_ROOT = Path(os.environ.get("XUVTOP", DATA_ROOT / "chianti")).expanduser()
CHIANTI_DATABASE_URL = os.environ.get("CHIANTI_DATABASE_URL", "")

if not CHIANTI_ROOT.exists() and CHIANTI_DATABASE_URL:
    archive_path = DATA_ROOT / Path(CHIANTI_DATABASE_URL).name
    print(f"Downloading CHIANTI database archive to {archive_path}")
    urllib.request.urlretrieve(CHIANTI_DATABASE_URL, archive_path)
    extract_root = DATA_ROOT / "chianti-extracted"
    if extract_root.exists():
        shutil.rmtree(extract_root)
    extract_root.mkdir()
    with tarfile.open(archive_path) as archive:
        archive.extractall(extract_root)
    candidates = [path for path in extract_root.rglob("VERSION")]
    if not candidates:
        raise RuntimeError("The CHIANTI archive did not contain a VERSION file.")
    CHIANTI_ROOT = candidates[0].parent

required_chianti_paths = [
    CHIANTI_ROOT / "VERSION",
    CHIANTI_ROOT / "masterlist" / "masterlist_ions.pkl",
    CHIANTI_ROOT / "h" / "h_1" / "h_1.elvlc",
    CHIANTI_ROOT / "he" / "he_1" / "he_1.elvlc",
    CHIANTI_ROOT / "he" / "he_2" / "he_2.elvlc",
]
missing = [path for path in required_chianti_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Set XUVTOP to an extracted CHIANTI database root or set CHIANTI_DATABASE_URL. "
        f"Missing: {missing}"
    )

os.environ["XUVTOP"] = str(CHIANTI_ROOT)
print(f"Using CHIANTI database: {CHIANTI_ROOT}")
print((CHIANTI_ROOT / "VERSION").read_text().strip())

Using CHIANTI database: /tmp/chianti
10.0


## Create Carsus readers

Import `carsus` before importing `astropy.units` or `astropy.constants` elsewhere in the kernel. Carsus pins the Astropy constants set used by legacy TARDIS atom-data generation during import.

In [2]:
from carsus.io.chianti_ import ChiantiReader
from carsus.io.kurucz import GFALLReader
from carsus.io.nist import NISTIonizationEnergies, NISTWeightsComp
from carsus.io.nuclear import NNDCReader
from carsus.io.zeta import KnoxLongZeta

atomic_weights = NISTWeightsComp()
ionization_energies = NISTIonizationEnergies("H-Zn")

# GFALLReader's default source is the Carsus Kurucz CD23 mirror on the main branch.
# Include labels in the level identity so same-energy/J terms remain distinct when labels differ.
gfall_reader = GFALLReader(
    "H-Zn",
    unique_level_identifier=["energy", "j", "label"],
)

chianti_reader = ChiantiReader("H-He", collisions=True, priority=20)
zeta_data = KnoxLongZeta()

# With remote=True, NNDCReader clones the Carsus NNDC CSV repository into its
# default location if it is not already present.
nndc_reader = NNDCReader(remote=True)

[ carsus.io.nist.weightscomp][   INFO] - Downloading data from the carsus-dat-nist repository (weightscomp.py:77)


[                py.warnings][WARNING] - /home/runner/micromamba/envs/carsus/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'raw.githubusercontent.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
 (_py_warnings.py:230)


[                py.warnings][WARNING] - /home/runner/micromamba/envs/carsus/lib/python3.14/site-packages/uncertainties/core.py:1024: UserWarning: Using UFloat objects with std_dev==0 may give unexpected results.
  warn("Using UFloat objects with std_dev==0 may give unexpected results.")
 (_py_warnings.py:230)


[  carsus.io.nist.ionization][   INFO] - Downloading ionization energies from the carsus-data-nist repo. (ionization.py:91)


[                py.warnings][WARNING] - /home/runner/micromamba/envs/carsus/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'raw.githubusercontent.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
 (_py_warnings.py:230)


...

[                py.warnings][WARNING] - /home/runner/micromamba/envs/carsus/lib/python3.14/site-packages/ChiantiPy/core/IpyMspectrum.py:104: SyntaxWarning: "\i" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\i"? A raw string is also an option.
  ylabel += '($\int\,$ N$_e\,$N$_H\,$d${\it l}$)$^{-1}$'
 (_py_warnings.py:230)


Cloning into '/home/runner/Downloads/carsus-data-nndc'...


[     carsus.io.nuclear.nndc][   INFO] - Downloading NNDC decay data from https://github.com/tardis-sn/carsus-data-nndc (nndc.py:42)


[     carsus.io.nuclear.nndc][   INFO] - Parsing decay data from: /home/runner/Downloads/carsus-data-nndc/csv (nndc.py:50)


Updating files: 100% (4651/4651), done.


## Build and write the TARDIS atom-data file

In [3]:
from carsus.io.output import TARDISAtomData

atom_data = TARDISAtomData(
    atomic_weights=atomic_weights,
    ionization_energies=ionization_energies,
    gfall_reader=gfall_reader,
    zeta_data=zeta_data,
    chianti_reader=chianti_reader,
    nndc_reader=nndc_reader,
)

atom_data.to_hdf(
    OUTPUT_PATH,
    legacy_tardis_schema=True,
    database_version="v0.9",
)
print(f"Wrote {OUTPUT_PATH.resolve()}")

[carsus.io.output.levels_lines][   INFO] - Ingesting energy levels. (levels_lines.py:167)


[     carsus.io.kurucz.gfall][   INFO] - Parsing GFALL from: https://github.com/tardis-sn/carsus-data-kurucz/raw/main/linelists/gfall/gfall.dat?raw=true (gfall.py:173)


[carsus.io.output.levels_lines][   INFO] - GFALL selected species: Li 0, Li 1, Be 0, Be 1, Be 2, B 0, B 1, B 2, B 3, C 0, C 1, C 2, C 3, N 0, N 1, N 2, N 3, N 4, N 5, O 0, O 1, O 2, O 3, O 4, O 5, F 0, F 1, F 2, F 3, F 4, F 5, Ne 0, Ne 1, Ne 2, Ne 3, Ne 4, Ne 5, Na 0, Na 1, Na 2, Na 3, Na 4, Na 5, Mg 0, Mg 1, Mg 2, Mg 3, Mg 4, Mg 5, Al 0, Al 1, Al 2, Al 3, Al 4, Al 5, Si 0, Si 1, Si 2, Si 3, Si 4, Si 5, P 0, P 1, P 2, P 3, P 4, P 5, S 0, S 1, S 2, S 3, S 4, S 5, Cl 0, Cl 1, Cl 2, Cl 3, Cl 4, Ar 0, Ar 1, Ar 2, Ar 3, Ar 4, K 0, K 1, K 2, K 3, K 4, Ca 0, Ca 1, Ca 2, Ca 3, Ca 4, Ca 5, Ca 6, Ca 7, Ca 8, Sc 0, Sc 1, Sc 2, Sc 3, Sc 4, Sc 5, Sc 6, Sc 7, Sc 8, Ti 0, Ti 1, Ti 2, Ti 3, Ti 4, Ti 5, Ti 6, Ti 7, Ti 8, V 0, V 1, V 2, V 3, V 4, V 5, V 6, V 7, V 8, Cr 0, Cr 1, Cr 2, Cr 3, Cr 4, Cr 5, Cr 6, Cr 7, Cr 8, Mn 0, Mn 1, Mn 2, Mn 3, Mn 4, Mn 5, Mn 6, Mn 7, Mn 8, Fe 0, Fe 1, Fe 2, Fe 3, Fe 4, Fe 5, Fe 6, Fe 7, Fe 8, Co 0, Co 1, Co 2, Co 3, Co 4, Co 5, Co 6, Co 7, Co 8, Ni 0, Ni 1, Ni 2, Ni 3, N

[carsus.io.output.levels_lines][   INFO] - Chianti selected species: H 0, He 0, He 1. (levels_lines.py:207)


[carsus.io.output.levels_lines][   INFO] - Ingesting transition lines. (levels_lines.py:297)


...

[      carsus.io.output.base][   INFO] - Signing TARDISAtomData. (base.py:284)


[      carsus.io.output.base][   INFO] - Format Version: 2.0 (base.py:285)


[      carsus.io.output.base][   INFO] - MD5: 4367f8f076fc685a1912258808f482e0 (base.py:286)


[      carsus.io.output.base][   INFO] - UUID1: 356c0f249b1f11f185787c1e52bf8172 (base.py:287)


Wrote /home/runner/work/carsus/carsus/docs/kurucz_cd23_chianti_H_He_latest.h5


## Validate the file layout

These checks verify the legacy TARDIS schema contract used by the Carsus regression test for `kurucz_cd23_chianti_H_He_latest.h5`.

In [4]:
import pandas as pd

expected_keys = {
    "/atom_data",
    "/collisions_data",
    "/collisions_metadata",
    "/decay_radiation_data",
    "/ionization_data",
    "/levels_data",
    "/lines_data",
    "/macro_atom_data",
    "/macro_atom_references",
    "/metadata",
    "/zeta_data",
}

with pd.HDFStore(OUTPUT_PATH, mode="r") as store:
    keys = set(store.keys())
    assert keys == expected_keys, sorted(keys ^ expected_keys)
    assert store.root._v_attrs["FORMAT_VERSION"] == "2.0"
    assert store.root._v_attrs["database_version"] == "v0.9"

    metadata = store["metadata"]
    assert ("md5sum", "levels") in metadata.index
    assert ("md5sum", "lines") in metadata.index
    assert ("md5sum", "levels_data") not in metadata.index
    assert ("md5sum", "lines_data") not in metadata.index

    atom_data_table = store["atom_data"]
    assert atom_data_table.index.name == "atomic_number"
    assert atom_data_table.index.min() == 1
    assert atom_data_table.index.max() == 94

    print("Validated legacy TARDIS atom-data schema.")
    print("Rows:")
    for key in sorted(keys):
        obj = store[key]
        shape = getattr(obj, "shape", None)
        print(f"  {key}: {shape}")

Validated legacy TARDIS atom-data schema.
Rows:
  /atom_data: (94, 3)
  /collisions_data: (358, 14)
  /collisions_metadata: (2,)


  /decay_radiation_data: (242295, 41)
  /ionization_data: (465,)
  /levels_data: (24805, 3)
  /lines_data: (271771, 8)
  /macro_atom_data: (815313, 7)
  /macro_atom_references: (24805, 3)
  /metadata: (18, 1)
  /zeta_data: (412, 20)


## Record source versions

The file metadata stores checksums and software versions. The printed values below are useful when publishing or uploading the generated atom-data file.

In [5]:
with pd.HDFStore(OUTPUT_PATH, mode="r") as store:
    print("HDF root attributes:")
    for name in ["FORMAT_VERSION", "database_version", "DATE", "MD5", "UUID1"]:
        print(f"  {name}: {store.root._v_attrs[name]}")

    print("\nMetadata:")
    display(store["metadata"])

print(f"CHIANTI XUVTOP: {os.environ['XUVTOP']}")
print(f"CHIANTI version: {(Path(os.environ['XUVTOP']) / 'VERSION').read_text().strip()}")
print(f"NNDC source directory: {nndc_reader.dirname}")
print(f"GFALL checksum: {gfall_reader.version}")

HDF root attributes:
  FORMAT_VERSION: 2.0
  database_version: v0.9
  DATE: 2026-08-18T16:09:36.778295+00:00
  MD5: 4367f8f076fc685a1912258808f482e0
  UUID1: 356c0f249b1f11f185787c1e52bf8172

Metadata:


value
field    key                                                    
format   version                                             2.0
md5sum   atom_data              3f0298f7de8c5a5015da18caa5863502
         collisions_data        90d9496cd6d2bb00bf8afa3a05daff5d
         collisions_metadata    33a56c0abaec5ec37473c130e9e68ff6
         decay_radiation_data   567484a1aff18ccb8b8afce542a14f6c
         ionization_data        d65d9bc36d1d5c947dbf47d241cd6c04
         levels                 836ad6a21e0103fbd4f7ace366d8b287
         lines                  0c246189c59986ec9da4e4bc53f60d31
         macro_atom_data        995bc5d237aa7eba1e2693407b562a78
         macro_atom_references  89588159b5c0330cd8ab787a65ade767
         zeta_data              3555b5c9000ae58eb92ac5d0018ca360
software python                                           3.14.5
         carsus                      2024.12.24.dev57+g0c5fa0595
         astropy                                           7.2.0
         numpy                                             2.4.6
         pandas                                            3.0.3
         tables                                           3.11.1
         ChiantiPy                                        0.16.0

CHIANTI XUVTOP: /tmp/chianti
CHIANTI version: 10.0
NNDC source directory: /home/runner/Downloads/carsus-data-nndc/csv
GFALL checksum: 2704fbda0b8cba61bb70426234224464
